# Filter training
In this notebook we train the filter with the labeled data produced in the previous notebook.  First, we embed the concatenated title+description via `FinBERT`, and then we cross-validate the following models:
- a simple logistic regression (baseline),
- a cosine-kNN model,
- a shallow neural network with 1 hidden layers with 8/16/32 nodes.

### Data loading and train-test split
We first load the labeled data

In [1]:
import pandas as pd

df = pd.read_csv("data/labeled_headlines.csv", index_col=False)
df = df[df.relevant != -1]

Then we embed each sentence

In [2]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("ProsusAI/finbert")
df["embeddings"] = embedder.encode(df.concat.to_list(), show_progress_bar=True).tolist()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/67 [00:00<?, ?it/s]

and produce a train-test split:

In [3]:
from sklearn.model_selection import train_test_split
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    df.embeddings, df.relevant, shuffle=True, random_state=42
)

# dataframe versions for sklearn
X_train_df = pd.DataFrame(X_train.to_list())
X_test_df = pd.DataFrame(X_test.to_list())

# numpy version for keras
X_train_np = np.stack(X_train.values)
X_test_np = np.stack(X_test.values)
y_train_np = y_train.values
y_test_np = y_test.values

### Models definitions
##### Basic logistic regression (baseline)

In [4]:
from sklearn.linear_model import LogisticRegressionCV

logit = LogisticRegressionCV(cv=5)

##### $k$-nearest neighbors

In [5]:
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=5, metric="cosine", weights="distance")

##### Shallow neural networks

In [6]:
import os

os.environ["KERAS_BACKEND"] = "torch"

import keras

keras.utils.set_random_seed(42)


def generate_shallow(nodes: int, summarize: bool = False) -> keras.Model:
    inputs = keras.Input(shape=(embedder.config.hidden_size,))
    x = keras.layers.Dense(nodes, activation="relu")(inputs)
    x = keras.layers.Dropout(0.5)(x)
    outputs = keras.layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(
        inputs=inputs, outputs=outputs, name=f"relevance_filter ({nodes} nodes)"
    )
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    if summarize:
        model.summary()

    return model

In [7]:
nn8 = generate_shallow(8)
nn16 = generate_shallow(16)
nn32 = generate_shallow(32)
nn64 = generate_shallow(64)

### Model selection

In [8]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

callback = keras.callbacks.EarlyStopping(patience=2, monitor="val_accuracy")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

for train_index, holdout_index in kf.split(X_train):
    # pandas
    X_tt_df = X_train_df.iloc[train_index]
    y_tt_df = y_train.iloc[train_index]
    X_th_df = X_train_df.iloc[holdout_index]
    y_th_df = y_train.iloc[holdout_index]

    # numpy
    X_tt_np = X_train_np[train_index]
    X_th_np = X_train_np[holdout_index]
    y_tt_np = y_train_np[train_index]
    y_th_np = y_train_np[holdout_index]

    scores = {}

    # logistic regression
    logit.fit(X_tt_df, y_tt_df)
    scores["logit"] = accuracy_score(y_th_df, logit.predict(X_th_df))

    # kNN
    neigh.fit(X_tt_df, y_tt_df)
    scores["kNN"] = accuracy_score(y_th_df, neigh.predict(X_th_df))

    # nn8
    nn8.fit(
        X_tt_np,
        y_tt_np,
        batch_size=64,
        epochs=15,
        validation_split=0.2,
        callbacks=[callback],
        verbose=0,
    )
    scores["nn8"] = accuracy_score(y_th_np, np.rint(nn8.predict(X_th_np)))

    # nn16
    nn16.fit(
        X_tt_np,
        y_tt_np,
        batch_size=64,
        epochs=15,
        validation_split=0.2,
        callbacks=[callback],
        verbose=0,
    )
    scores["nn16"] = accuracy_score(y_th_np, np.rint(nn16.predict(X_th_np, verbose=0)))

    # nn32
    nn32.fit(
        X_tt_np,
        y_tt_np,
        batch_size=64,
        epochs=15,
        validation_split=0.2,
        callbacks=[callback],
        verbose=0,
    )
    scores["nn32"] = accuracy_score(y_th_np, np.rint(nn32.predict(X_th_np, verbose=0)))

    # nn64
    nn64.fit(
        X_tt_np,
        y_tt_np,
        batch_size=64,
        epochs=15,
        validation_split=0.2,
        callbacks=[callback],
        verbose=0,
    )
    scores["nn64"] = accuracy_score(y_th_np, np.rint(nn64.predict(X_th_np, verbose=0)))

    results.append(scores)

results = pd.DataFrame(results)

/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 977us/step


/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [9]:
results

,logit,kNN,nn8,nn16,nn32,nn64
0,0.943750,0.928125,0.956250,0.928125,0.931250,0.931250
1,0.943750,0.940625,0.946875,0.925000,0.943750,0.946875
2,0.959375,0.934375,0.968750,0.940625,0.950000,0.968750
3,0.940625,0.918750,0.950000,0.956250,0.937500,0.959375
4,0.956250,0.909375,0.950000,0.953125,0.953125,0.959375


In [10]:
summary = pd.concat([results.mean(), results.std()], keys=["mean", "std"], axis=1)
summary

,mean,std
logit,0.948750,0.008443
kNN,0.926250,0.012422
nn8,0.954375,0.008728
nn16,0.940625,0.014149
nn32,0.943125,0.008949
nn64,0.953125,0.014490


The logistic regression is already a great baseline. However, it might be worth considering also `nn16` and `nn32`.

In [11]:
# logit
logit.fit(X_train_df, y_train)

# nn16
nn16.fit(
    X_train_np,
    y_train_np,
    batch_size=64,
    epochs=15,
    validation_split=0.2,
    callbacks=[callback],
    verbose=0,
)

# summary
print()
print("Logit accuracy score:", accuracy_score(y_test, logit.predict(X_test_df)))
print(
    "nn16 accuracy score: ",
    accuracy_score(y_test_np, np.rint(nn16.predict(X_test_np, verbose=0))),
)

/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(



Logit accuracy score: 0.949438202247191
nn16 accuracy score:  0.9325842696629213


Since we don't have much data, we will use logistic regression: it's a simple model with good accuracy not prone to overfitting.  When we will have more data, we might implement more sophisticated models.